In [8]:
import pickle

import numpy as np
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1. Load Data
df = pd.read_parquet('../data/green_tripdata_2024-01.parquet')

# 2. Compute Target & Filter Outliers (Duration & Distance)
df['duration'] = (df.lpep_dropoff_datetime - df.lpep_pickup_datetime).dt.total_seconds() / 60
df = df[(df.duration >= 1) & (df.duration <= 60)].copy()
df = df[(df.trip_distance > 0) & (df.trip_distance <= 100)].copy()

# 3. Clean & Combine Categorical Features
df['PULocationID'] = df['PULocationID'].fillna(-1).astype(int)
df['DOLocationID'] = df['DOLocationID'].fillna(-1).astype(int)
df['PU_DO'] = df['PULocationID'].astype(str) + '_' + df['DOLocationID'].astype(str)

categorical = ['PU_DO']
numerical = ['trip_distance']

# 4. Train/Validation Split (80/20)
train_size = int(len(df) * 0.8)
df_train = df.iloc[:train_size]
df_val = df.iloc[train_size:]

# 5. Vectorize Features
dicts_train = df_train[categorical + numerical].to_dict(orient='records')
dicts_val = df_val[categorical + numerical].to_dict(orient='records')

dv = DictVectorizer()
X_train = dv.fit_transform(dicts_train)
X_val = dv.transform(dicts_val)

y_train = df_train['duration'].values
y_val = df_val['duration'].values

# 6. Fit Stable Model (Ridge)
model = Ridge(alpha=1.0)
model.fit(X_train, y_train)

# 7. Evaluate Metrics
y_pred = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
mae = mean_absolute_error(y_val, y_pred)

print(f"Validation RMSE: {rmse:.4f}")
print(f"Validation MAE: {mae:.4f}")

# 8. Save Model & Vectorizer
with open('../models/baseline.pkl', 'wb') as f_out:
    pickle.dump((dv, model), f_out)

Validation RMSE: 5.8897
Validation MAE: 3.7349
